# core

> Ergonomic wrapper for pandas_gbq that simplifies loading BigQuery data into DataFrames

This module provides an ergonomic wrapper around `pandas_gbq` to simplify working with BigQuery in pandas. The main functions are:

- `read()` - Load data from BigQuery queries or tables into DataFrames with automatic type conversion
- `to()` - Write DataFrames to BigQuery tables
- `ex()` - Execute queries without returning results (useful for DDL/DML)
- `exists()` - Check if a table exists
- `ensure()` - Read a table if it exists, otherwise create it from a query first

All functions include verbose timing and size reporting by default.

In [ ]:
#| default_exp core

**Imports**

In [ ]:
from nbdev.showdoc import *

In [ ]:
#| export
from pandas_gbq import read_gbq as _original_read_gbq, to_gbq as _original_to_gbq, Context, context
import pandas as pd, re, time
from decimal import Decimal
from google.cloud import bigquery
from dataclasses import dataclass
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
from google.oauth2 import service_account
from google.auth import default
import os
import json

**Credentials helper** - Get credentials from environment variable for CI/CD, or fall back to ADC

In [ ]:
#| export
def get_creds():
    "Get credentials and project_id from environment variable or ADC"
    creds_json = os.getenv('GOOGLE_CREDENTIALS_JSON')
    if creds_json:
        creds = service_account.Credentials.from_service_account_info(json.loads(creds_json))
        return creds, creds.project_id
    creds, proj = default()
    proj = os.getenv('GCP_PROJECT') or os.getenv('GOOGLE_CLOUD_PROJECT') or proj
    if not proj: raise ValueError("Could not determine project_id. Set GCP_PROJECT or GOOGLE_CLOUD_PROJECT env var.")
    return creds, proj

def _creds_proj(
    credentials=None,  # Google credentials object, or None to use defaults
    project_id=None    # GCP project ID, or None to auto-detect
):
    "Resolve credentials and project_id, using defaults if not provided"
    if credentials and project_id: return credentials, project_id
    creds, proj = get_creds()
    return credentials or creds, project_id or proj


**Core read function** - Wraps `pandas_gbq.read_gbq` with automatic type conversion and timing

In [ ]:
#| export
def read(
    query_or_table:str, # BigQuery SQL query or table reference
    verbose:bool=True, # Print timing and size info
    convert_dtypes:bool=True, # Convert to pandas nullable types
    date_cols:list=None, # Additional columns to convert to datetime
    str_cols:list=["scv_id"], # Columns to keep as string/object type
    use_bqstorage_api:bool=True, # Use Storage API for faster downloads
    credentials=None, # Google credentials (defaults to ADC)
    project_id=None, # GCP project ID (defaults to credential's project)
    **kwargs
):
    "Load data from BigQuery query or table into DataFrame"
    start = time.time()
    creds, proj = _creds_proj(credentials, project_id)
    is_table = re.match(r"""^[`\w\-]+\.[\w\-]+\.[\w\-\`]+$""", query_or_table)
    query = f"SELECT * FROM `{query_or_table.replace('`','')}`" if is_table else query_or_table
    df = _original_read_gbq(query, project_id=proj, use_bqstorage_api=use_bqstorage_api, credentials=creds, **kwargs)
    if convert_dtypes: df = convert_bq_dtypes(df, date_cols=date_cols, str_cols=str_cols)
    if verbose:
        elapsed = time.time() - start
        size_gb = _get_size_gb(df)
        print(f"Loaded {len(df)} rows × {len(df.columns)} cols ({size_gb:.4f} GB) from {'table' if is_table else 'query'} in {elapsed:.2f}s")
    return df


**Type conversion helper** - Converts BigQuery types to appropriate pandas nullable types

In [ ]:
#| export
def convert_bq_dtypes(
    df:pd.DataFrame, # DataFrame to convert
    date_cols:list=None, # List of columns to convert to datetime
    str_cols:list=None # List of columns to keep as string/object type
):
    "Convert BigQuery data types to pandas-compatible types"
    df = df.copy()
    date_cols = set(date_cols or [])
    str_cols = set(str_cols or [])
    for col in df.columns:
        if col in str_cols: df[col] = df[col].astype('object')
        elif re.search(r"(date|timestamp)", col.lower()) or col in date_cols: df[col] = pd.to_datetime(df[col], errors='coerce')
        elif df[col].dtype == 'float64': df[col] = df[col].astype('Float64')
        elif df[col].dtype == 'int64': df[col] = df[col].astype('Int64')
        elif df[col].dtype == 'object':
            first_val = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
            if first_val is not None and isinstance(first_val, Decimal): df[col] = df[col].astype('Float64')
    return df

**Write function** - Wraps `pandas_gbq.to_gbq` with timing info

In [ ]:
#| export
def to(
    df:pd.DataFrame,
    destination_table:str,
    verbose:bool=True,
    credentials=None,
    project_id=None,
    **kwargs
):
    "Write DataFrame to BigQuery table"
    start = time.time()
    creds, proj = _creds_proj(credentials, project_id)
    result = _original_to_gbq(df, destination_table, project_id=proj, credentials=creds, **kwargs)
    if verbose:
        elapsed = time.time() - start
        size_gb = _get_size_gb(df)
        print(f"Sent {len(df)} rows × {len(df.columns)} cols ({size_gb:.4f} GB) to {destination_table} in {elapsed:.2f}s")
    return result

**Memory helper** - Calculate DataFrame size in GB

In [ ]:
#| export

def _get_size_gb(df):
    return df.memory_usage(deep=True).sum() / 1024**3


**Execute function** - Run queries without returning results (DDL/DML operations)

In [ ]:
#| export
def ex(query:str, project_id:str=None, verbose:bool=True, credentials=None, **kwargs):
    "Execute query in BigQuery without returning results"
    creds, proj = _creds_proj(credentials, project_id)
    client = bigquery.Client(project=proj, credentials=creds, **kwargs)
    start = time.time()
    job = client.query(query)
    result = job.result()
    if verbose:
        elapsed = time.time() - start
        gb_processed = (job.total_bytes_processed or 0) / 1024**3
        rows_affected = job.num_dml_affected_rows if job.num_dml_affected_rows else 0
        cached = " (cached)" if job.cache_hit else ""
        print(f"Processed {gb_processed:.4f} GB{cached}, {rows_affected} rows affected in {elapsed:.2f}s")
    return result

**Table helper class** - Parse and represent BigQuery table references

In [ ]:
#| export
@dataclass
class Table:
    project: str
    dataset: str
    name: str
    
    @classmethod
    def from_id(cls, table_id:str):
        parts = table_id.replace('`','').split('.')
        if len(parts) == 3: return cls(*parts)
        if len(parts) == 2: return cls(os.getenv('GCP_PROJECT'), *parts)
        raise ValueError(f"Invalid table_id: {table_id}")
    
    @property
    def id(self): return f"{self.project}.{self.dataset}.{self.name}"

    def __str__(self): return self.id



**Existence checker** - Check if a BigQuery table exists

In [ ]:
#| export
def exists(table_id:str, project_id:str=None, credentials=None, verbose:bool=False, **kwargs):
    "Check if a BigQuery table exists"
    creds, proj = _creds_proj(credentials, project_id)
    if isinstance(table_id, Table): table_id = table_id.id
    elif table_id.count('.') == 1: table_id = f"{proj}.{table_id}"
    client = bigquery.Client(credentials=creds, project=proj, **kwargs)
    try:
        client.get_table(table_id)
        if verbose: print(f"Table {table_id} exists")
        return True
    except NotFound:
        if verbose: print(f"Table {table_id} not found")
        return False

**Ensure function** - Read table if exists, otherwise create from query then read

In [ ]:
#| export
def ensure(table_id:str, query:str, project_id:str=None, force:bool=False, verbose:bool=True, credentials=None, **kwargs):
    "Read table if exists, otherwise create from query then read"
    creds, proj = _creds_proj(credentials, project_id)
    if not exists(table_id, project_id=proj, credentials=creds, verbose=verbose) or force:
        create_query = f"CREATE OR REPLACE TABLE {table_id} AS {query}"
        ex(create_query, project_id=proj, credentials=creds, verbose=verbose)
    return read(table_id, verbose=verbose, credentials=creds, project_id=proj, **kwargs)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
